# Computer Systems Data: Exploratory Analysis

This notebook performs exploratory data analysis on the computer systems dataset to understand its structure, patterns, and relationships.

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import sys

# Add parent directory to path for local imports
sys.path.append('..')

# Set plotting style
plt.style.use('seaborn-whitegrid')
sns.set_context("notebook", font_scale=1.2)

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

## 1. Load Raw Data

We'll load the raw computer systems dataset and examine its structure.

In [ ]:
# Load the data using the same encoding as in the preprocessing module
raw_data_path = '../data/db_computers_2025_raw.csv'

# Check if file exists
if os.path.exists(raw_data_path):
    df_raw = pd.read_csv(raw_data_path, encoding='utf-8-sig', low_memory=False)
    print(f"Loaded data: {df_raw.shape[0]} rows, {df_raw.shape[1]} columns")
else:
    print(f"File not found: {raw_data_path}")
    print("Please place the 'db_computers_2025_raw.csv' file in the data directory.")

In [ ]:
# Examine basic information about the dataset
display(df_raw.head())
print("\nDataset Information:")
display(df_raw.info())

## 2. Data Quality Assessment

Let's examine data quality issues such as missing values and inconsistencies.

In [ ]:
# Check for missing values
missing = pd.DataFrame({
    'Missing Count': df_raw.isna().sum(),
    'Missing Percentage': (df_raw.isna().sum() / len(df_raw) * 100).round(2)
}).sort_values('Missing Count', ascending=False)

display(missing[missing['Missing Count'] > 0])

# Visualize missing values
plt.figure(figsize=(12, 8))
missing_top = missing[missing['Missing Count'] > 0].head(15)
sns.barplot(x=missing_top.index, y='Missing Percentage', data=missing_top)
plt.xticks(rotation=90)
plt.title('Missing Values by Feature (Top 15)', fontsize=16)
plt.tight_layout()
plt.show()

In [ ]:
# Examine data types and potential inconsistencies
print("Data Types:")
display(df_raw.dtypes.value_counts())

# Examine numeric columns that might have text mixed in
numeric_columns = ['Price', 'Storage', 'RAM', 'Screen Size', 'Processor Speed', 'Weight']
for col in numeric_columns:
    if col in df_raw.columns:
        print(f"\nSample values from {col}:")
        display(df_raw[col].sample(5))

## 3. Descriptive Statistics

Let's calculate and visualize summary statistics for key features.

In [ ]:
# Import preprocessing module to clean numeric columns
from src.preprocessing import clean_numeric_with_units

# Clean numeric columns for analysis
df_cleaned = df_raw.copy()
for col in numeric_columns:
    if col in df_cleaned.columns:
        df_cleaned = clean_numeric_with_units(df_cleaned, col)

# Calculate summary statistics for numeric columns
numeric_df = df_cleaned[numeric_columns].dropna()
display(numeric_df.describe())

In [ ]:
# Create histograms for numeric features
fig, axes = plt.subplots(nrows=2, ncols=3, figsize=(18, 10))
axes = axes.flatten()

for i, col in enumerate(numeric_columns):
    if i < len(axes) and col in numeric_df.columns:
        sns.histplot(numeric_df[col], kde=True, ax=axes[i])
        axes[i].set_title(f'Distribution of {col}')
        axes[i].set_xlabel(col)
        axes[i].set_ylabel('Frequency')

plt.tight_layout()
plt.show()

In [ ]:
# Create box plots for numeric features
fig, axes = plt.subplots(nrows=2, ncols=3, figsize=(18, 10))
axes = axes.flatten()

for i, col in enumerate(numeric_columns):
    if i < len(axes) and col in numeric_df.columns:
        sns.boxplot(y=numeric_df[col], ax=axes[i])
        axes[i].set_title(f'Boxplot of {col}')
        axes[i].set_ylabel(col)

plt.tight_layout()
plt.show()

## 4. Categorical Features Analysis

Let's examine the categorical features in our dataset.

In [ ]:
# Identify categorical columns
categorical_columns = ['Brand', 'OS', 'Type', 'Processor', 'GPU']

# Analyze categorical features
for col in categorical_columns:
    if col in df_raw.columns:
        print(f"\nValue counts for {col}:")
        display(df_raw[col].value_counts().head(10))
        
        # Visualize top categories
        plt.figure(figsize=(12, 6))
        top_categories = df_raw[col].value_counts().head(10)
        sns.barplot(x=top_categories.index, y=top_categories.values)
        plt.title(f'Top 10 {col} Categories')
        plt.xticks(rotation=45, ha='right')
        plt.tight_layout()
        plt.show()

## 5. Relationship Analysis

Let's examine relationships between key features, particularly with respect to price.

In [ ]:
# Correlation matrix for numeric features
plt.figure(figsize=(10, 8))
correlation = numeric_df.corr()
mask = np.triu(np.ones_like(correlation, dtype=bool))
sns.heatmap(correlation, mask=mask, annot=True, cmap='coolwarm', vmin=-1, vmax=1, fmt='.2f')
plt.title('Correlation Matrix')
plt.tight_layout()
plt.show()

# Display correlations with Price
if 'Price' in numeric_df.columns:
    price_corr = correlation['Price'].sort_values(ascending=False)
    print("\nCorrelations with Price:")
    display(price_corr)

In [ ]:
# Scatterplots for Price vs other numeric features
if 'Price' in numeric_df.columns:
    for col in numeric_columns:
        if col != 'Price' and col in numeric_df.columns:
            plt.figure(figsize=(10, 6))
            sns.scatterplot(x=col, y='Price', data=numeric_df, alpha=0.6)
            plt.title(f'Price vs {col}')
            plt.tight_layout()
            plt.show()

In [ ]:
# Price distributions by categories
if 'Price' in df_cleaned.columns:
    for col in categorical_columns:
        if col in df_cleaned.columns:
            plt.figure(figsize=(12, 6))
            top_categories = df_cleaned[col].value_counts().head(6).index
            category_data = df_cleaned[df_cleaned[col].isin(top_categories)]
            
            sns.boxplot(x=col, y='Price', data=category_data)
            plt.title(f'Price Distribution by {col}')
            plt.xticks(rotation=45, ha='right')
            plt.tight_layout()
            plt.show()

## 6. Clustering Analysis

Let's perform K-means clustering on the numeric features and evaluate using silhouette scores.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# Prepare data for clustering
cluster_df = numeric_df.dropna()

# Scale features
scaler = StandardScaler()
scaled_features = scaler.fit_transform(cluster_df)

# Evaluate different values of k
silhouette_scores = []
inertia = []

for k in range(2, 11):
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(scaled_features)
    cluster_labels = kmeans.labels_
    
    # Calculate silhouette score
    silhouette_avg = silhouette_score(scaled_features, cluster_labels)
    silhouette_scores.append(silhouette_avg)
    
    # Calculate inertia (sum of squared distances)
    inertia.append(kmeans.inertia_)
    
    print(f"For k={k}, the silhouette score is {silhouette_avg:.3f}")

In [ ]:
# Plot silhouette scores
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(range(2, 11), silhouette_scores, 'o-')
plt.xlabel('Number of Clusters (k)')
plt.ylabel('Silhouette Score')
plt.title('Silhouette Score vs. Number of Clusters')
plt.grid(True)

# Plot elbow method
plt.subplot(1, 2, 2)
plt.plot(range(2, 11), inertia, 'o-')
plt.xlabel('Number of Clusters (k)')
plt.ylabel('Inertia')
plt.title('Elbow Method')
plt.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
# Determine optimal k value
optimal_k = silhouette_scores.index(max(silhouette_scores)) + 2  # +2 because we started from k=2
print(f"Optimal number of clusters based on silhouette score: {optimal_k}")

# Perform clustering with optimal k
kmeans = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)
cluster_labels = kmeans.fit_predict(scaled_features)

# Add cluster labels to the original data
cluster_df['Cluster'] = cluster_labels

# Examine cluster centers
cluster_centers = pd.DataFrame(
    scaler.inverse_transform(kmeans.cluster_centers_), 
    columns=cluster_df.columns[:-1]
)
cluster_centers.index.name = 'Cluster'

print("\nCluster Centers:")
display(cluster_centers)

In [ ]:
# Visualize clusters
if len(numeric_columns) >= 2:
    plt.figure(figsize=(12, 8))
    
    # Select first two features for visualization
    x_feature = numeric_columns[0]
    y_feature = numeric_columns[1]
    
    # Plot data points colored by cluster
    sns.scatterplot(x=x_feature, y=y_feature, hue='Cluster', data=cluster_df, palette='viridis', s=60, alpha=0.7)
    
    # Plot cluster centers
    plt.scatter(
        cluster_centers[x_feature], 
        cluster_centers[y_feature], 
        s=200, 
        marker='X', 
        c='red', 
        label='Cluster Centers'
    )
    
    plt.title(f'Clusters in {x_feature} vs {y_feature} Space')
    plt.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
# Analyze clusters
cluster_analysis = cluster_df.groupby('Cluster').agg(['mean', 'count'])
display(cluster_analysis)

# Distribution of clusters
plt.figure(figsize=(10, 6))
cluster_df['Cluster'].value_counts().sort_index().plot(kind='bar')
plt.title('Number of Items per Cluster')
plt.xlabel('Cluster')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

## 7. Summary and Key Findings

Document key findings and insights from the exploratory analysis.

Based on our exploratory data analysis, here are the key findings:

1. **Data Quality:**
   - The dataset contains missing values that need to be handled appropriately during preprocessing.
   - Numeric columns contain text (units) that need to be cleaned and standardized.
   - Some features may have multiple versions (e.g., Screen Size, Display Size) that should be merged.

2. **Features Importance:**
   - RAM, Storage, and Processor Speed show strong correlations with Price.
   - Brand and Type also significantly influence pricing.

3. **Clustering:**
   - The optimal number of clusters appears to be [based on the silhouette scores].
   - The clusters represent distinct computer system categories with different price points and specifications.

4. **Recommendations for Modeling:**
   - Focus on key features with strong price correlations.
   - Handle categorical features with appropriate encoding techniques.
   - Consider using non-linear models to capture complex relationships.